In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import duckdb


In [2]:
# Load master parquet file with merged NYISO load and weather data
# Path from: CS506_Project/3_OUTPUT/3_svr/svr_copy/ to CS506_Project/1_LIB/master/master.parquet
from pathlib import Path

MASTER_PATH = Path("../../../1_LIB/master/master.parquet")
print(f"Loading data from: {MASTER_PATH}")

# Load the master parquet file
df = pd.read_parquet(MASTER_PATH)

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")



Loading data from: ..\..\..\1_LIB\master\master.parquet
Loaded 1057304 rows
Columns: ['datetime', 'latitude [degrees_north]', 'longitude [degrees_east]', 'elevation [feet]', 'temp_2m [degF]', 'temp_9m [degF]', 'apparent_temperature [degF]', 'relative_humidity [percent]', 'dewpoint [degF]', 'precip_incremental [inch]', 'precip_local [inch]', 'precip_max_intensity [inch/hour]', 'precip_1hr [inch]', 'avg_wind_speed_prop [mile/hr]', 'max_wind_speed_prop [mile/hr]', 'wind_speed_stddev_prop [mile/hr]', 'wind_direction_prop [degrees]', 'wind_direction_stddev_prop [degrees]', 'avg_wind_speed_sonic [mile/hr]', 'max_wind_speed_sonic [mile/hr]', 'wind_speed_stddev_sonic [mile/hr]', 'wind_direction_sonic [degrees]', 'wind_direction_stddev_sonic [degrees]', 'avg_wind_speed_merge [mile/hr]', 'max_wind_speed_merge [mile/hr]', 'wind_speed_stddev_merge [mile/hr]', 'wind_direction_merge [degrees]', 'wind_direction_stddev_merge [degrees]', 'solar_insolation [W/m^2]', 'station_pressure [inHg]', 'frozen_so

In [3]:
# Standardize datetime column name
if 'datetime' in df.columns:
    df['Time'] = pd.to_datetime(df['datetime'], utc=True)
elif 'Time Stamp' in df.columns:
    df['Time'] = pd.to_datetime(df['Time Stamp'], utc=True)

df.head()

,datetime,latitude [degrees_north],longitude [degrees_east],elevation [feet],temp_2m [degF],temp_9m [degF],apparent_temperature [degF],relative_humidity [percent],dewpoint [degF],precip_incremental [inch],...,soil_temp_05cm [degF],soil_temp_25cm [degF],soil_temp_50cm [degF],soil_moisture_05cm [m^3/m^3],soil_moisture_25cm [m^3/m^3],soil_moisture_50cm [m^3/m^3],snow_depth [inch],PTID,Load,Time
0,2015-08-10 04:00:00+00:00,43.117,-73.57829,103,NaN,NaN,NaN,95.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61757.0,1613.772727,2015-08-10 04:00:00+00:00
1,2015-08-10 04:05:00+00:00,43.117,-73.57829,103,NaN,NaN,NaN,95.6,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61757.0,1602.554545,2015-08-10 04:05:00+00:00
2,2015-08-10 04:10:00+00:00,43.117,-73.57829,103,NaN,NaN,NaN,96.6,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61757.0,1592.854545,2015-08-10 04:10:00+00:00
3,2015-08-10 04:15:00+00:00,43.117,-73.57829,103,NaN,NaN,NaN,96.7,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61757.0,1585.218182,2015-08-10 04:15:00+00:00
4,2015-08-10 04:20:00+00:00,43.117,-73.57829,103,NaN,NaN,NaN,96.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61757.0,1574.563636,2015-08-10 04:20:00+00:00


In [4]:
df

,datetime,latitude [degrees_north],longitude [degrees_east],elevation [feet],temp_2m [degF],temp_9m [degF],apparent_temperature [degF],relative_humidity [percent],dewpoint [degF],precip_incremental [inch],...,soil_temp_05cm [degF],soil_temp_25cm [degF],soil_temp_50cm [degF],soil_moisture_05cm [m^3/m^3],soil_moisture_25cm [m^3/m^3],soil_moisture_50cm [m^3/m^3],snow_depth [inch],PTID,Load,Time
0,2015-08-10 04:00:00+00:00,43.11700,-73.57829,103,NaN,NaN,NaN,95.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61757.0,1613.772727,2015-08-10 04:00:00+00:00
1,2015-08-10 04:05:00+00:00,43.11700,-73.57829,103,NaN,NaN,NaN,95.6,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61757.0,1602.554545,2015-08-10 04:05:00+00:00
2,2015-08-10 04:10:00+00:00,43.11700,-73.57829,103,NaN,NaN,NaN,96.6,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61757.0,1592.854545,2015-08-10 04:10:00+00:00
3,2015-08-10 04:15:00+00:00,43.11700,-73.57829,103,NaN,NaN,NaN,96.7,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61757.0,1585.218182,2015-08-10 04:15:00+00:00
4,2015-08-10 04:20:00+00:00,43.11700,-73.57829,103,NaN,NaN,NaN,96.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,61757.0,1574.563636,2015-08-10 04:20:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1057299,2025-10-01 03:35:00+00:00,42.04036,-77.23726,1665,50.6,53.2,50.6,66.4,39.9,0.0,...,59.9,61.9,61.9,0.22,0.28,0.24,NaN,61757.0,1387.435145,2025-10-01 03:35:00+00:00
1057300,2025-10-01 03:40:00+00:00,42.04036,-77.23726,1665,50.9,53.3,50.9,65.0,39.6,0.0,...,59.9,61.9,61.9,0.22,0.28,0.24,NaN,61757.0,1372.123527,2025-10-01 03:40:00+00:00
1057301,2025-10-01 03:45:00+00:00,42.04036,-77.23726,1665,50.1,52.9,50.1,66.3,39.3,0.0,...,59.7,61.9,61.9,0.22,0.28,0.24,NaN,61757.0,1375.259982,2025-10-01 03:45:00+00:00
1057302,2025-10-01 03:50:00+00:00,42.04036,-77.23726,1665,50.6,52.4,50.6,65.6,39.5,0.0,...,59.7,61.8,61.9,0.22,0.27,0.24,NaN,61757.0,1370.948655,2025-10-01 03:50:00+00:00


In [5]:
# Aggregate load by timestamp (sum across all PTIDs/zones)
df_total_load = df.groupby("Time", as_index=False)["Load"].sum()
print(f"Aggregated to {len(df_total_load)} unique timestamps")


Aggregated to 1057304 unique timestamps


In [6]:
df_total_load

,Time,Load
0,2015-08-10 04:00:00+00:00,1613.772727
1,2015-08-10 04:05:00+00:00,1602.554545
2,2015-08-10 04:10:00+00:00,1592.854545
3,2015-08-10 04:15:00+00:00,1585.218182
4,2015-08-10 04:20:00+00:00,1574.563636
...,...,...
1057299,2025-10-01 03:35:00+00:00,1387.435145
1057300,2025-10-01 03:40:00+00:00,1372.123527
1057301,2025-10-01 03:45:00+00:00,1375.259982
1057302,2025-10-01 03:50:00+00:00,1370.948655


In [7]:
# Weather features are already in master parquet
# Aggregate weather features by timestamp (average across stations/PTIDs)
weather_cols = [
    'temp_2m [degF]',
    'apparent_temperature [degF]',
    'relative_humidity [percent]',
    'precip_1hr [inch]',
    'avg_wind_speed_merge [mile/hr]',
    'solar_insolation [W/m^2]'
]

# Check which weather columns exist
existing_weather_cols = [col for col in weather_cols if col in df.columns]
print(f"Available weather features: {existing_weather_cols}")


Available weather features: ['temp_2m [degF]', 'apparent_temperature [degF]', 'relative_humidity [percent]', 'precip_1hr [inch]', 'avg_wind_speed_merge [mile/hr]', 'solar_insolation [W/m^2]']


In [8]:
# Aggregate weather data by timestamp
agg_dict = {col: 'mean' for col in existing_weather_cols}
agg_dict['Load'] = 'sum'

df_aggregated = df.groupby('Time', as_index=False).agg(agg_dict)
print(f"Aggregated data shape: {df_aggregated.shape}")
print(df_aggregated.head())


Aggregated data shape: (1057304, 8)
                       Time  temp_2m [degF]  apparent_temperature [degF]  \
0 2015-08-10 04:00:00+00:00             NaN                          NaN   
1 2015-08-10 04:05:00+00:00             NaN                          NaN   
2 2015-08-10 04:10:00+00:00             NaN                          NaN   
3 2015-08-10 04:15:00+00:00             NaN                          NaN   
4 2015-08-10 04:20:00+00:00             NaN                          NaN   

   relative_humidity [percent]  precip_1hr [inch]  \
0                         95.2                NaN   
1                         95.6                NaN   
2                         96.6                NaN   
3                         96.7                NaN   
4                         96.5                NaN   

   avg_wind_speed_merge [mile/hr]  solar_insolation [W/m^2]         Load  
0                             0.0                       0.0  1613.772727  
1                             0.0     

In [9]:
# Rename columns to match expected format
rename_dict = {
    'temp_2m [degF]': 'avg_temp',
    'apparent_temperature [degF]': 'avg_apparent_temp',
    'relative_humidity [percent]': 'avg_humidity',
    'precip_1hr [inch]': 'total_precip',
    'avg_wind_speed_merge [mile/hr]': 'avg_wind_speed',
    'solar_insolation [W/m^2]': 'avg_solar'
}

df_final = df_aggregated.rename(columns=rename_dict)
print(df_final.head())
print(f"\nFinal dataset shape: {df_final.shape}")


                       Time  avg_temp  avg_apparent_temp  avg_humidity  \
0 2015-08-10 04:00:00+00:00       NaN                NaN          95.2   
1 2015-08-10 04:05:00+00:00       NaN                NaN          95.6   
2 2015-08-10 04:10:00+00:00       NaN                NaN          96.6   
3 2015-08-10 04:15:00+00:00       NaN                NaN          96.7   
4 2015-08-10 04:20:00+00:00       NaN                NaN          96.5   

   total_precip  avg_wind_speed  avg_solar         Load  
0           NaN             0.0        0.0  1613.772727  
1           NaN             0.0        0.0  1602.554545  
2           NaN             0.0        0.0  1592.854545  
3           NaN             0.0        0.0  1585.218182  
4           NaN             0.0        0.0  1574.563636  

Final dataset shape: (1057304, 8)


In [10]:
print(df_final.describe())

           avg_temp  avg_apparent_temp  avg_humidity   total_precip  \
count  1.050273e+06       1.050176e+06  1.034901e+06  945685.000000   
mean   4.776645e+01       4.526034e+01  7.457726e+01       0.004439   
std    1.860279e+01       2.136041e+01  1.840994e+01       0.029321   
min   -2.100000e+01      -3.130000e+01  1.310000e+01       0.000000   
25%    3.290000e+01       2.830000e+01  6.140000e+01       0.000000   
50%    4.840000e+01       4.660000e+01  7.720000e+01       0.000000   
75%    6.320000e+01       6.320000e+01  9.080000e+01       0.000000   
max    9.250000e+01       9.740000e+01  1.000000e+02       1.880000   

       avg_wind_speed     avg_solar          Load  
count    1.050724e+06  1.050758e+06  1.057304e+06  
mean     5.895009e+00  1.446819e+02  1.600469e+03  
std      4.067733e+00  2.325785e+02  2.989744e+02  
min      0.000000e+00  0.000000e+00  0.000000e+00  
25%      2.800000e+00  0.000000e+00  1.386073e+03  
50%      5.300000e+00  4.000000e+00  1.564440e+0

In [11]:
# Check for missing values
print("Missing values per column:")
print(df_final.isnull().sum())


Missing values per column:
Time                      0
avg_temp               7031
avg_apparent_temp      7128
avg_humidity          22403
total_precip         111619
avg_wind_speed         6580
avg_solar              6546
Load                      0
dtype: int64


In [12]:
# Data is already at 5-minute intervals from master parquet
# Resample to 15-minute intervals if needed
df_final['Time'] = pd.to_datetime(df_final['Time'])
df_final = df_final.set_index('Time')

# Resample to 15-minute intervals
df_15min = df_final.resample('15min').mean().reset_index()
df_15min = df_15min.dropna()

print(f"15-minute aggregated data shape: {df_15min.shape}")
print(df_15min.head())



15-minute aggregated data shape: (311396, 8)
                         Time   avg_temp  avg_apparent_temp  avg_humidity  \
127 2015-08-11 11:45:00+00:00  65.300000          65.300000     93.466667   
128 2015-08-11 12:00:00+00:00  65.100000          65.100000     96.633333   
129 2015-08-11 12:15:00+00:00  65.100000          65.100000     97.033333   
130 2015-08-11 12:30:00+00:00  65.300000          65.300000     97.333333   
131 2015-08-11 12:45:00+00:00  65.433333          65.433333     96.633333   

     total_precip  avg_wind_speed  avg_solar         Load  
127      0.089000        0.066667  19.000000  1871.800000  
128      0.085000        0.000000  18.000000  1897.984848  
129      0.135333        0.033333  21.666667  1926.115152  
130      0.152667        0.300000  30.000000  1941.896970  
131      0.164667        0.166667  26.666667  1954.575758  


In [13]:
# Data is already merged from master parquet
merged = df_15min.copy()
print(f"Merged data columns: {merged.columns.tolist()}")


Merged data columns: ['Time', 'avg_temp', 'avg_apparent_temp', 'avg_humidity', 'total_precip', 'avg_wind_speed', 'avg_solar', 'Load']


In [14]:
merged

,Time,avg_temp,avg_apparent_temp,avg_humidity,total_precip,avg_wind_speed,avg_solar,Load
127,2015-08-11 11:45:00+00:00,65.300000,65.300000,93.466667,0.089000,0.066667,19.000000,1871.800000
128,2015-08-11 12:00:00+00:00,65.100000,65.100000,96.633333,0.085000,0.000000,18.000000,1897.984848
129,2015-08-11 12:15:00+00:00,65.100000,65.100000,97.033333,0.135333,0.033333,21.666667,1926.115152
130,2015-08-11 12:30:00+00:00,65.300000,65.300000,97.333333,0.152667,0.300000,30.000000,1941.896970
131,2015-08-11 12:45:00+00:00,65.433333,65.433333,96.633333,0.164667,0.166667,26.666667,1954.575758
...,...,...,...,...,...,...,...,...
355675,2025-10-01 02:45:00+00:00,50.833333,50.833333,69.466667,0.000000,1.933333,0.000000,1469.066688
355676,2025-10-01 03:00:00+00:00,50.166667,50.166667,72.666667,0.000000,2.833333,0.000000,1441.573924
355677,2025-10-01 03:15:00+00:00,50.400000,50.266667,69.733333,0.000000,3.400000,0.000000,1414.154203
355678,2025-10-01 03:30:00+00:00,50.733333,50.733333,66.766667,0.000000,4.233333,0.000000,1386.337245


In [15]:
# Check the final data before training
merged.info()
merged.head()

<class 'pandas.core.frame.DataFrame'>
Index: 311396 entries, 127 to 355679
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype              
---  ------             --------------   -----              
 0   Time               311396 non-null  datetime64[ns, UTC]
 1   avg_temp           311396 non-null  float64            
 2   avg_apparent_temp  311396 non-null  float64            
 3   avg_humidity       311396 non-null  float64            
 4   total_precip       311396 non-null  float64            
 5   avg_wind_speed     311396 non-null  float64            
 6   avg_solar          311396 non-null  float64            
 7   Load               311396 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(7)
memory usage: 21.4 MB


,Time,avg_temp,avg_apparent_temp,avg_humidity,total_precip,avg_wind_speed,avg_solar,Load
127,2015-08-11 11:45:00+00:00,65.300000,65.300000,93.466667,0.089000,0.066667,19.000000,1871.800000
128,2015-08-11 12:00:00+00:00,65.100000,65.100000,96.633333,0.085000,0.000000,18.000000,1897.984848
129,2015-08-11 12:15:00+00:00,65.100000,65.100000,97.033333,0.135333,0.033333,21.666667,1926.115152
130,2015-08-11 12:30:00+00:00,65.300000,65.300000,97.333333,0.152667,0.300000,30.000000,1941.896970
131,2015-08-11 12:45:00+00:00,65.433333,65.433333,96.633333,0.164667,0.166667,26.666667,1954.575758


In [16]:
df_total_load = merged

In [17]:
# Split the data based on the year
train_data = df_total_load[df_total_load['Time'].dt.year.between(2001, 2021)]
val_data = df_total_load[df_total_load['Time'].dt.year == 2022]
test_data = df_total_load[df_total_load['Time'].dt.year.isin([2023, 2024, 2025])]

# Print the sizes of each split
print(f"Training data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")
print(f"Testing data size: {len(test_data)}")

Training data size: 182009
Validation data size: 34954
Testing data size: 94433


In [18]:
train_data

,Time,avg_temp,avg_apparent_temp,avg_humidity,total_precip,avg_wind_speed,avg_solar,Load
127,2015-08-11 11:45:00+00:00,65.300000,65.300000,93.466667,0.089000,0.066667,19.000000,1871.800000
128,2015-08-11 12:00:00+00:00,65.100000,65.100000,96.633333,0.085000,0.000000,18.000000,1897.984848
129,2015-08-11 12:15:00+00:00,65.100000,65.100000,97.033333,0.135333,0.033333,21.666667,1926.115152
130,2015-08-11 12:30:00+00:00,65.300000,65.300000,97.333333,0.152667,0.300000,30.000000,1941.896970
131,2015-08-11 12:45:00+00:00,65.433333,65.433333,96.633333,0.164667,0.166667,26.666667,1954.575758
...,...,...,...,...,...,...,...,...
224235,2021-12-31 22:45:00+00:00,40.700000,40.700000,88.433333,0.000000,0.600000,0.000000,1677.090909
224236,2021-12-31 23:00:00+00:00,40.533333,40.533333,88.500000,0.000000,1.600000,0.000000,1666.612997
224237,2021-12-31 23:15:00+00:00,40.333333,40.333333,89.033333,0.000000,1.100000,0.000000,1658.992452
224238,2021-12-31 23:30:00+00:00,40.866667,40.866667,89.000000,0.000000,1.633333,0.000000,1647.980182


In [19]:
train_data = train_data.dropna()
val_data = val_data.dropna()
test_data = test_data.dropna()
scaler = StandardScaler()
y_scaler = StandardScaler()

# Drop Time and Load columns for features
feature_cols = [col for col in train_data.columns if col not in ['Time', 'Load', 'index']]
print(f"Feature columns: {feature_cols}")

train_scaled = scaler.fit_transform(train_data[feature_cols])
val_scaled = scaler.transform(val_data[feature_cols])
test_scaled = scaler.transform(test_data[feature_cols])

# Use ravel() to convert to 1D array for scikit-learn compatibility
y_train_scaled = y_scaler.fit_transform(train_data[['Load']]).ravel()
y_val_scaled = y_scaler.transform(val_data[['Load']]).ravel()
y_test_scaled = y_scaler.transform(test_data[['Load']]).ravel()


Feature columns: ['avg_temp', 'avg_apparent_temp', 'avg_humidity', 'total_precip', 'avg_wind_speed', 'avg_solar']


In [20]:
def create_dataset(X, y, time_steps=1):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        v = X[i:i + time_steps]
        Xs.append(v)
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

TIME_STEPS = 5
X_train, y_train = create_dataset(train_scaled, y_train_scaled, TIME_STEPS)
X_val, y_val = create_dataset(val_scaled, y_val_scaled, TIME_STEPS)
X_test, y_test = create_dataset(test_scaled, y_test_scaled, TIME_STEPS)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(182004, 5, 6) (182004,)
(34949, 5, 6) (34949,)
(94428, 5, 6) (94428,)


In [21]:
print("NaNs in X_train:", np.isnan(X_train).sum())
print("NaNs in X_val:", np.isnan(X_val).sum())
print("NaNs in y_train:", np.isnan(y_train).sum())
print("NaNs in y_val:", np.isnan(y_val).sum())

NaNs in X_train: 0
NaNs in X_val: 0
NaNs in y_train: 0
NaNs in y_val: 0


In [22]:
subset_size = 40000
val_subset = 8749
X_train_sub = X_train[:subset_size]
y_train_sub = y_train[:subset_size]
X_val_sub = X_val[:val_subset]
y_val_sub = y_val[:val_subset]


In [23]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
# best_mae = float('inf')
# best_model = None

# for C in [0.1, 1, 10]:
#     for gamma in ['scale', 0.01, 0.001]:
#         for epsilon in [0.01, 0.1, 0.5, 1.0]:
#             print(f"C={C}, gamma={gamma}, epsilon={epsilon}")
#             model = SVR(kernel='rbf', C=C, gamma=gamma, epsilon=epsilon)
#             model.fit(X_train_sub.reshape(X_train_sub.shape[0], -1), y_train_sub)
#             preds = model.predict(X_val_sub.reshape(X_val_sub.shape[0], -1))
#             mae = mean_absolute_error(y_val_sub, preds)
#             print(f"val MAE={mae:.3f}")

#             if mae < best_mae:
#                 best_mae = mae
#                 best_model = model

# print("Best params found:", best_model.get_params())


Best params found: {'C': 10, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}

Best mae: 0.04994650252799576

In [ ]:
best_params = {'C': 10, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}
best_model = SVR(**best_params)
best_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)

In [ ]:

# Make predictions
train_pred = best_model.predict(X_train.reshape(X_train.shape[0], -1))
train_pred = y_scaler.inverse_transform(train_pred.reshape(-1, 1))
y_train_inv = y_scaler.inverse_transform(y_train.reshape(-1, 1))

val_pred = best_model.predict(X_val.reshape(X_val.shape[0], -1))
val_pred = y_scaler.inverse_transform(val_pred.reshape(-1, 1))
y_val_inv = y_scaler.inverse_transform(y_val.reshape(-1, 1))

test_pred = best_model.predict(X_test.reshape(X_test.shape[0], -1))
test_pred = y_scaler.inverse_transform(test_pred.reshape(-1, 1))
y_test_inv = y_scaler.inverse_transform(y_test.reshape(-1, 1))


In [ ]:
# Evaluate the model
mae_train = mean_absolute_error(y_train_inv, train_pred)
mae_test = mean_absolute_error(y_test_inv, test_pred)
print("Mean Absolute Error on Training Data:", mae_train)
print("Mean Absolute Error on Testing Data:", mae_test)

In [ ]:

from sklearn.metrics import mean_absolute_percentage_error


mape_train = mean_absolute_percentage_error(y_train_inv, train_pred)
mape_test = mean_absolute_percentage_error(y_test_inv, test_pred)
print("Mean Absolute Error on Training Data:", mape_train)
print("Mean Absolute Error on Testing Data:", mape_test)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100 

y_true = y_test_inv
y_pred = test_pred
x = np.arange(len(y_true))

window = 100  # how many points to show at once
lag = 1       # number of steps prediction lags behind

fig, ax = plt.subplots(figsize=(10,6))
line_true, = ax.plot([], [], label="True Values", color="blue", alpha=0.7)
line_pred, = ax.plot([], [], label="Predictions", color="red", alpha=0.7)
ax.set_ylim(min(y_true.min(), y_pred.min())*0.95, max(y_true.max(), y_pred.max())*1.05)
ax.set_xlabel("Index")
ax.set_ylabel("Load")
ax.set_title("Predictions vs True Values (Trailing Window)")
ax.legend()

def update(frame):
    start = max(0, frame - window)
    end = frame
    line_true.set_data(x[start:end], y_true[start:end])
    
    # Prediction lags behind true values
    pred_start = max(0, frame - window - lag)
    pred_end = max(0, frame - lag)
    line_pred.set_data(x[pred_start:pred_end], y_pred[pred_start:pred_end])
    
    ax.set_xlim(x[start], x[end-1] if end > start else x[start]+1)
    return line_true, line_pred

ani = FuncAnimation(
    fig, update,
    frames=range(0, len(x), 10),   # every 10th frame
    interval=20, blit=True
)
HTML(ani.to_jshtml())



In [ ]:
mape = np.mean(np.abs((y_test_inv - test_pred) / y_test_inv)) * 100
print(f"Testing MAPE: {mape:.2f}%")

eps = 1e-6
mape = np.mean(np.abs((y_train_inv - train_pred) / (y_train_inv + eps))) * 100
print(f"Training MAPE: {mape:.2f}%")

mape = np.mean(np.abs((y_val_inv - val_pred) / y_val_inv)) * 100
print(f"Val MAPE: {mape:.2f}%")



In [ ]:
from sklearn.metrics import r2_score

rmse = np.sqrt(mean_squared_error(y_test_inv, test_pred))
print(f"Testing RMSE: {rmse}")

r2 = r2_score(y_test_inv, test_pred)
print(f"Testing R2: {r2}")

In [ ]:
import joblib

joblib.dump(best_model, "svr_min15_model.joblib")


In [ ]:
subset_start = 0
subset_end = 1000
plt.figure(figsize=(12,6))
plt.plot(y_test_inv[subset_start: subset_end], label="True Load", color="black", alpha=0.7)
plt.plot(test_pred[subset_start: subset_end], label="Predictions", color="orange", alpha=0.7)
plt.xlabel("Time Index")
plt.ylabel("Load (MW)")
plt.title("SVR Predictions vs True Load")
plt.legend()
plt.show()

In [ ]:
# from datetime import datetime
# import matplotlib.pyplot as plt
# import meteostat
# from meteostat import Point, Daily, Hourly

# start = datetime(2018, 1, 1, 0, 0)
# end = datetime(2018, 1, 1, 12, 0)

# # Create Point for Vancouver, BC
# vancouver = Point(42.65, -73.75)

# # Get daily data for 2018
# data = Hourly(vancouver, start, end)
# data = data.fetch()

# # Plot line chart including average, minimum and maximum temperature
# data